# Reproduce Experiment 2b — GJW with SYK scrambling

Retrieves job `d7h873q2khts739peivg` (11 couplings × ⟨X⟩, ⟨Y⟩, ⟨Z⟩ on the output qubit) and recomputes the fidelity with Alice's state.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np

service = QiskitRuntimeService(channel="ibm_cloud")

def check(name, value, paper, tol):
    ok = abs(value - paper) <= tol
    print(f"{name:32s} recomputed {value:9.4f}   paper {paper:9.4f}   {'OK' if ok else 'DIFFERENT'}")

In [ ]:
from qiskit.quantum_info import DensityMatrix, state_fidelity

def random_qubit(seed=99):
    rng = np.random.default_rng(seed)
    psi = rng.normal(0, 1, 2) + 1j * rng.normal(0, 1, 2)
    return psi / np.linalg.norm(psi)

PSI_ALICE = random_qubit(99)
X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Z = np.array([[1, 0], [0, -1]])

job = service.job("d7h873q2khts739peivg")
r = job.result()
F = []
for i in range(11):
    ex, ey, ez = (float(r[3*i + k].data.evs) for k in range(3))
    rho = DensityMatrix((np.eye(2) + ex*X + ey*Y + ez*Z) / 2)
    F.append(float(np.real(state_fidelity(DensityMatrix(PSI_ALICE), rho, validate=False))))
for g, f in zip(np.linspace(0, np.pi/2, 11), F):
    print(f"g/pi = {g/np.pi:.2f}   F = {f:.4f}")

In [ ]:
check("F(g=0)", F[0], 0.020, 0.0005)
check("F(g=pi/2)", F[-1], 0.953, 0.0005)
check("GJW gain", F[-1] - F[0], 0.933, 0.0005)